# 04b_qwen_pca_sweep — Qwen3 Embeddings at Multiple PCA Dims (Colab)

Same Qwen3-0.6B embeddings as `04_text_features_qwen`, but this notebook saves
the embeddings reduced to **several PCA dimensions (50 / 100 / 200 / 300)** so we
can test whether keeping more components improves multiple prediction.

The embedding is generated **once**; PCA is then run several times (fast).

**Before running**
1. `MyDrive/workshop-2026/text_private_YYYY-MM-DD.csv` already uploaded.
2. Runtime -> GPU (T4).

**Output** (one file per dimension)
- `text_embeddings_qwen_pca50_YYYY-MM-DD.csv`
- `text_embeddings_qwen_pca100_YYYY-MM-DD.csv`
- `text_embeddings_qwen_pca200_YYYY-MM-DD.csv`
- `text_embeddings_qwen_pca300_YYYY-MM-DD.csv`

> Note: 2,848 rows. 200-300 PCA dims is large for this size -> watch for
> overfitting when comparing in 06b (CV R2 may plateau or drop).


## 1. Install & Mount

In [ ]:
!pip install -q -U "sentence-transformers>=2.7.0" "transformers>=4.51.0"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/workshop-2026')
print('Drive folder exists?:', DRIVE_DIR.exists())
for p in sorted(DRIVE_DIR.glob('*.csv')):
    print('  -', p.name)


## 2. Load & Prepare Text

Identical text preparation to the other 04 notebooks.


In [ ]:
import ast
import pandas as pd

text_files = sorted(DRIVE_DIR.glob('text_private_*.csv'))
assert text_files, 'No text_private_*.csv found. Upload it first.'
df = pd.read_csv(text_files[-1])
print('Loaded:', text_files[-1].name, '->', df.shape)

def join_list(x):
    try:
        lst = ast.literal_eval(x) if isinstance(x, str) else x
        if isinstance(lst, list):
            return ' '.join(str(i) for i in lst)
    except Exception:
        pass
    return str(x) if pd.notna(x) else ''

for col in ['opportunities', 'risks']:
    if col in df.columns:
        df[f'{col}_text'] = df[col].apply(join_list)

combined = df['summary'].fillna('').astype(str)
for col in ['opportunities_text', 'risks_text']:
    if col in df.columns:
        combined = combined + ' ' + df[col].fillna('').astype(str)
df['combined_text'] = combined.str.strip()
print('Rows:', len(df))


## 3. Generate Embeddings Once (Qwen3-0.6B)

Generate the full 1024-dim embeddings a single time. We reuse them for every PCA size.


In [ ]:
from sentence_transformers import SentenceTransformer
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

model = SentenceTransformer('Qwen/Qwen3-Embedding-0.6B', device=device)
embeddings = model.encode(
    df['combined_text'].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
)
print('Full embeddings shape:', embeddings.shape)   # (2848, 1024)


## 4. PCA Sweep — 50 / 100 / 200 / 300

Fit one PCA up to the largest dimension, then slice it for the smaller ones.
Slicing the same fitted PCA keeps the smaller sets as exact prefixes of the
larger ones, so the comparison is clean.


In [ ]:
from sklearn.decomposition import PCA
import numpy as np

DIMS = [50, 100, 200, 300]
max_dim = min(max(DIMS), embeddings.shape[0], embeddings.shape[1])

pca = PCA(n_components=max_dim)
reduced_max = pca.fit_transform(embeddings)
cumvar = np.cumsum(pca.explained_variance_ratio_)

print(f'Fitted PCA to {max_dim} components.')
for d in DIMS:
    if d <= max_dim:
        print(f'  {d:3d} dims -> explains {cumvar[d-1]*100:.1f}% of variance')


## 5. Save One File per Dimension

Each file uses the same `text_emb_i` column naming so 06b can load any of them
with identical code. Filenames encode the PCA size.


In [ ]:
import datetime
today = datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%d')

for d in DIMS:
    if d > max_dim:
        print(f'Skip {d}: exceeds available components ({max_dim})')
        continue
    reduced = reduced_max[:, :d]
    emb_df = pd.DataFrame(reduced, columns=[f'text_emb_{i}' for i in range(d)])
    emb_df.insert(0, 'id', df['id'].values)
    out_path = DRIVE_DIR / f'text_embeddings_qwen_pca{d}_{today}.csv'
    emb_df.to_csv(out_path, index=False)
    print(f'Saved pca{d}: {out_path.name}  shape={emb_df.shape}')


## 6. Next

- Download all four `text_embeddings_qwen_pca*_*.csv` files to local `data/processed/`.
- Compare them with `06c_compare_pca.py` (a variant of 06b that sweeps PCA dims).
- Expect: R2 may rise from 50 to ~100 dims, then plateau or **drop** at 200-300
  as overfitting sets in on 2,848 rows. The turning point is the finding.
